<a href="https://colab.research.google.com/github/Borwec/ida_25_26/blob/main/lab4/lab4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset
from collections import Counter
import torch.nn.functional as F
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import random

torch.manual_seed(127)
random.seed(127)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Device: cuda


In [2]:
!wget https://www.manythings.org/anki/ukr-eng.zip
!unzip -o ukr-eng.zip

--2025-12-30 19:54:04--  https://www.manythings.org/anki/ukr-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4914352 (4.7M) [application/zip]
Saving to: ‘ukr-eng.zip’

ukr-eng.zip         100%[===================>]   4.69M  13.5MB/s    in 0.3s    

2025-12-30 19:54:05 (13.5 MB/s) - ‘ukr-eng.zip’ saved [4914352/4914352]

Archive:  ukr-eng.zip
  inflating: ukr.txt                 
  inflating: _about.txt              


In [3]:
lines = open("ukr.txt", encoding="utf-8").read().split("\n")
pairs = []
for line in lines:
  parts = line.split("\t")
  if len(parts) < 2:
    continue
  eng = parts[0].strip()
  ukr = parts[1].strip()
  pairs.append((eng, ukr))

print(len(pairs))
for _ in range(5):
    print(random.choice(pairs))

160049
('These are mine.', 'Ці належать мені.')
('What makes you think Tom would ever do that?', 'Чому ти думаєш, що Том це коли-небудь робитиме?')
("Tom wanted to know who Mary's father was.", 'Том хотів знати, хто батько Мері.')
('She loves singing.', 'Вона любить співати.')
('They said no.', 'Том сказав "ні".')


In [4]:
def tokenize(text):
  return word_tokenize(text.lower().strip())

In [5]:
specials = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
MAX_VOCAB = 5_000
eng_stcs = [p[0] for p in pairs]
ukr_stcs = [p[1] for p in pairs]

def build_vocab(sentences):
  counter = Counter()

  for s in sentences:
    counter.update(tokenize(s))

  most_common = counter.most_common(MAX_VOCAB - len(specials))
  itos = specials + [w for w, _ in most_common]
  stoi = {w: i for i, w in enumerate(itos)}

  return stoi, itos

In [6]:
eng_stcs = [p[0] for p in pairs]
ukr_stcs = [p[1] for p in pairs]

eng_stoi, eng_itos = build_vocab(eng_stcs)
ukr_stoi, ukr_itos = build_vocab(ukr_stcs)

PAD_IDX = eng_stoi["[PAD]"]
UNK_IDX = eng_stoi["[UNK]"]
BOS_IDX = eng_stoi["[BOS]"]
EOS_IDX = eng_stoi["[EOS]"]

In [7]:
MAX_LEN_SRC = 20
MAX_LEN_TGT = 20

def encode(text, stoi, add_specials=False):
    tokens = tokenize(text)
    ids = []

    if add_specials:
        ids.append(BOS_IDX)

    for t in tokens:
        ids.append(stoi.get(t, UNK_IDX))

    if add_specials:
        ids.append(EOS_IDX)

    return ids

def prepare_pair(eng, ukr):
    src = encode(eng, eng_stoi, add_specials=False)

    tgt_full = encode(ukr, ukr_stoi, add_specials=True)
    tgt_in = tgt_full[:-1]
    tgt_out = tgt_full[1:]

    src = src[:MAX_LEN_SRC] + [PAD_IDX] * (MAX_LEN_SRC - len(src))
    tgt_in = tgt_in[:MAX_LEN_TGT] + [PAD_IDX] * (MAX_LEN_TGT - len(tgt_in))
    tgt_out = tgt_out[:MAX_LEN_TGT] + [PAD_IDX] * (MAX_LEN_TGT - len(tgt_out))

    return src, tgt_in, tgt_out

def encode_src_sentence(text, stoi, max_len):
    tokens = tokenize(text)
    ids = [eng_stoi.get(t, UNK_IDX) for t in tokens]
    ids = ids[:max_len] + [PAD_IDX] * (max_len - len(ids))
    return torch.tensor(ids, dtype=torch.long).unsqueeze(0)


def decode_tgt_ids(ids, itos):
    tokens = []
    for i in ids:
        if i in (PAD_IDX, BOS_IDX, EOS_IDX):
            continue
        tokens.append(itos[i])
    return " ".join(tokens)

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng, ukr = self.pairs[idx]
        src, tgt_in, tgt_out = prepare_pair(eng, ukr)

        return (
            torch.tensor(src, dtype=torch.long),
            torch.tensor(tgt_in, dtype=torch.long),
            torch.tensor(tgt_out, dtype=torch.long),
        )

def make_padding_mask(x, pad_idx=PAD_IDX):
    return (x == pad_idx)

def generate_subsequent_mask(size, device):
    mask = torch.triu(torch.ones(size, size, dtype=torch.bool, device=device), diagonal=1)
    return mask

class TokenPositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=50, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        x = self.token_emb(x) + self.pos_emb(pos)
        return self.dropout(x)

class Seq2SeqTransformer(nn.Module):
    def __init__(self,
                 src_vocab_size,
                 tgt_vocab_size,
                 d_model=64,
                 n_heads=4,
                 num_layers=2,
                 d_ff=128,
                 max_len_src=20,
                 max_len_tgt=20,
                 dropout=0.1):
        super().__init__()

        self.src_emb = TokenPositionalEmbedding(src_vocab_size, d_model, max_len_src, dropout)
        self.tgt_emb = TokenPositionalEmbedding(tgt_vocab_size, d_model, max_len_tgt, dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
        )

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
        )

        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)

        self.output_proj = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt_in):
        src_key_padding_mask = make_padding_mask(src)
        tgt_key_padding_mask = make_padding_mask(tgt_in)

        enc_in = self.src_emb(src)
        dec_in = self.tgt_emb(tgt_in)

        memory = self.encoder(
            enc_in,
            src_key_padding_mask=src_key_padding_mask
        )

        T = tgt_in.size(1)
        tgt_mask = generate_subsequent_mask(T, device=tgt_in.device)

        dec_out = self.decoder(
            dec_in,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )

        logits = self.output_proj(dec_out)
        return logits

In [8]:
src_vocab_size = len(eng_stoi)
tgt_vocab_size = len(ukr_stoi)

model = Seq2SeqTransformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=64,
    n_heads=4,
    num_layers=3,
    d_ff=128,
    max_len_src=MAX_LEN_SRC,
    max_len_tgt=MAX_LEN_TGT,
).to(device)

In [9]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [10]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    n_tokens = 0

    for src, tgt_in, tgt_out in loader:
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)

        optimizer.zero_grad()

        logits = model(src, tgt_in)
        B, T, V = logits.shape

        loss = criterion(
            logits.view(B*T, V),
            tgt_out.view(B*T)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * (tgt_out != PAD_IDX).sum().item()
        n_tokens += (tgt_out != PAD_IDX).sum().item()

    return total_loss / n_tokens

In [11]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    n_tokens = 0

    for src, tgt_in, tgt_out in loader:
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)

        logits = model(src, tgt_in)
        B, T, V = logits.shape

        loss = criterion(
            logits.view(B*T, V),
            tgt_out.view(B*T)
        )

        total_loss += loss.item() * (tgt_out != PAD_IDX).sum().item()
        n_tokens += (tgt_out != PAD_IDX).sum().item()

    return total_loss / n_tokens

In [12]:
@torch.no_grad()
def translate(model, sentence, max_len=MAX_LEN_TGT):
    model.eval()
    device = next(model.parameters()).device

    src = encode_src_sentence(sentence, eng_stoi, MAX_LEN_SRC).to(device)
    src_key_padding_mask = make_padding_mask(src)

    enc_in = model.src_emb(src)
    memory = model.encoder(enc_in, src_key_padding_mask=src_key_padding_mask)

    tgt_ids = [BOS_IDX]
    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_ids, dtype=torch.long, device=device).unsqueeze(0)
        tgt_key_padding_mask = make_padding_mask(tgt_tensor)

        T = tgt_tensor.size(1)
        tgt_mask = generate_subsequent_mask(T, device=device)

        dec_in = model.tgt_emb(tgt_tensor)
        dec_out = model.decoder(
            dec_in,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )

        logits = model.output_proj(dec_out[:, -1, :])
        next_id = logits.argmax(dim=-1).item()

        tgt_ids.append(next_id)

        if next_id == EOS_IDX:
            break

    translation = decode_tgt_ids(tgt_ids, ukr_itos)
    return translation

In [13]:
random.shuffle(pairs)

total = len(pairs)
train_size = int(0.95 * total)


train_pairs = pairs[:train_size]
val_pairs   = pairs[train_size: ]


print("Train:", len(train_pairs))
print("Val:", len(val_pairs))

print("Total:", len(train_pairs) + len(val_pairs))

Train: 152046
Val: 8003
Total: 160049


In [14]:
train_ds = TranslationDataset(train_pairs)
val_ds   = TranslationDataset(val_pairs)

BATCH_SIZE = 64

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

In [15]:
EPOCHS = 10

for ep in range(1, EPOCHS+1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {ep}/{EPOCHS} - train loss/token: {train_loss:.4f}  val loss/token: {val_loss:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1/10 - train loss/token: 2.9676  val loss/token: 1.9886
Epoch 2/10 - train loss/token: 1.8609  val loss/token: 1.4675
Epoch 3/10 - train loss/token: 1.5058  val loss/token: 1.2748
Epoch 4/10 - train loss/token: 1.3356  val loss/token: 1.1708
Epoch 5/10 - train loss/token: 1.2322  val loss/token: 1.1062
Epoch 6/10 - train loss/token: 1.1570  val loss/token: 1.0632
Epoch 7/10 - train loss/token: 1.0988  val loss/token: 1.0303
Epoch 8/10 - train loss/token: 1.0524  val loss/token: 0.9938
Epoch 9/10 - train loss/token: 1.0139  val loss/token: 0.9678
Epoch 10/10 - train loss/token: 0.9810  val loss/token: 0.9582


In [16]:
examples = [
    "The quick brown fox.",
    "Another brick in the wall.",
    "Through the fire and flames."
]

for s in examples:
    print("EN:", s)
    print("UA:", translate(model, s))
    print()

EN: The quick brown fox.
UA: [UNK] [UNK] .

EN: Another brick in the wall.
UA: ще [UNK] у стіні .

EN: Through the fire and flames.
UA: [UNK] вогонь та [UNK] .

